In [1]:
import numpy as np 
import pandas as pd
import casadi as ca

# Optimisation

**Anna Bergougnoux Santini et Julien Colella**  
Avril 2026  

---

1. Le revenu d'une entreprise s'écrit :  

   $$
   R(y)= p(y)y - c(y)
   $$

   Ici, la formule va de soi : on multiplie la matrice des quantités de matière première commandées $c=(c_i)$ avec la matrice coût unitaire $r=(r_i)$.

   $min(q,d)$ correspond à la quantité vendue :  
   - si $q > d$, comme on ne peut pas vendre plus que la demande, la quantité vendue est donc $d$  
   - si $q < d$, on vend une quantité $q$

2. Ce terme-ci n'est pas différentiable.

3. Soit $\alpha \gg 1$ :

4. On cherche alors à minimiser :

   $$
   g(q,d) = - v^T h(q, d) + c^T r
   $$

   sous la contrainte :

   $$
   Aq - r \le 0
   $$

   On peut donc réécrire ceci avec $z=(q,r)$ :

   $$
   g(z) = - v^T h(z_1, d) + c^T z_2
   $$

5. Ainsi que :

   $$
   c(z) = A z_1 - z_2
   $$

   $z$ dépend donc des deux vecteurs $q$ et $r$, et donc des $q_j$ et des $r_i$.




6. Pour ce problème, qui est donc un problème d'optimisation sous contrainte, où $f$ est différentiable non-linéaire et $c$ est une contrainte linéaire, on peut penser aux conditions de KKT.



**Question 6**

In [ ]:
alpha = 0.1
c = 1e-3 * np.array([30,1,1.3,4,1])
v = np.array([0.9,1.5,1.1])
d = np.array([400,67,33])
A = np.array([
    [3.5,2,1],
    [250,80,25],
    [0,8,3],
    [0,40,10],
    [0,8.5,0]
])

m = 5
p = 3

c = ca.DM(c)
v = ca.DM(v)
#on a eu un problème avec les multiplications entre array et objets casadi, 
#on a donc converti les array de facon à ce que les ojets manipulés soient compatibles

r = ca.MX.sym('r', m)
q = ca.MX.sym('q', p)


def h(q, d, alpha):
    return (q*ca.exp(-alpha*q) + d*ca.exp(-alpha*d)) / (ca.exp(-alpha*q) + ca.exp(-alpha*d))

f = c.T @ r - v.T @ h(q, d, alpha)


#contraintes
c1 = A @ q - r
c2 = -r
c3 = -q

c = ca.vertcat(c1, c2, c3)


nlp = {'x': ca.vertcat(r,q), 'f': f, 'g': c}

solver = ca.nlpsol('solver', 'ipopt', nlp)


sol = solver(lbg=-ca.inf, ubg=0)

z_opt = sol['x']
print(z_opt)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:       28
Number of nonzeros in Lagrangian Hessian.............:        3

Total number of variables............................:        8
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality c

Les résultats sont conformes à l'intuition : on commande juste assez pour satisfaire la demande avec un peu de marge, et on maximise ainsi la quantité vendue.

**Question 7**

*Question 7.a*

On souhaite maximiser l'**espérance** du revenu (3) :

$$\max_{q \geq 0,\, r \geq 0} \; \mathbb{E}\left[ v^T h(q, d) - c^T r \right] = \max_{q \geq 0,\, r \geq 0} \sum_{k=1}^{K} \pi^k \left( v^T h(q, d^k) - c^T r \right)$$

Puisque $r$ ne dépend pas du scénario, on peut factoriser :

$$\max_{q \geq 0,\, r \geq 0} \; \sum_{k=1}^{K} \pi^k \, v^T h(q, d^k) \;-\; c^T r$$

d'où le problème

$$\boxed{\max_{q \geq 0,\, r \geq 0} \; \sum_{k=1}^{K} \pi^k \, v^T h(q, d^k) - c^T r}$$

sous la contrainte :

$$r \geq Aq$$

où $h_i(q, d^k) = \dfrac{q_i e^{-\alpha q_i} + d^k_i e^{-\alpha d^k_i}}{e^{-\alpha q_i} + e^{-\alpha d^k_i}}$ est l'approximation du $\min$.

On peut aussi formuler ce problème comme un problème de minimisatiion, mais l'énoncé spécifie qu'on cherche à maximiser l'espérance du coût. Nous avons donc laissé cette formulation-ci.

*Question 7.b*

In [ ]:
alpha = 0.1
c_vec = 1e-3 * np.array([30,1,1.3,4,1])
v_vec = np.array([0.9,1.5,1.1])

d1 = np.array([400,67,33])
d2 = np.array([500,80,53])
d3 = np.array([300,60,43])
pi = np.array([0.5, 0.3, 0.2])
scenarios = [d1, d2, d3]

A = np.array([
    [3.5,2,1],
    [250,80,25],
    [0,8,3],
    [0,40,10],
    [0,8.5,0]
])

m = 5
p = 3

c_dm = ca.DM(c_vec)
v_dm = ca.DM(v_vec)

r = ca.MX.sym('r', m)
q = ca.MX.sym('q', p)

def h(q, d, alpha):
    return (q*ca.exp(-alpha*q) + d*ca.exp(-alpha*d)) / (ca.exp(-alpha*q) + ca.exp(-alpha*d))

# Objectif : espérance sur les K scénarios
esperance = sum(pi[k] * v_dm.T @ h(q, ca.DM(scenarios[k]), alpha) for k in range(3))
f = c_dm.T @ r - esperance

# Contraintes (inchangées)
c1 = A @ q - r
c2 = -r
c3 = -q
g = ca.vertcat(c1, c2, c3)

nlp = {'x': ca.vertcat(r, q), 'f': f, 'g': g}
solver = ca.nlpsol('solver', 'ipopt', nlp)

sol = solver(lbg=-ca.inf, ubg=0)
z_opt = sol['x']
print(z_opt)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:       28
Number of nonzeros in Lagrangian Hessian.............:        3

Total number of variables............................:        8
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality c

In [ ]:
alpha = 0.1
c = 1e-3 * np.array([30,1,1.3,4,1])
v = np.array([0.9,1.5,1.1])
d1 = np.array([400,67,33])
d2 = np.array([500,80,53])
d3 = np.array([300,60,43])
A = np.array([
    [3.5,2,1],
    [250,80,25],
    [0,8,3],
    [0,40,10],
    [0,8.5,0]
])

m = 5
p = 3

def h(q, d, alpha):
    return (q*ca.exp(-alpha*q) + d*ca.exp(-alpha*d)) / (ca.exp(-alpha*q) + ca.exp(-alpha*d))

c = ca.DM(c)
v = ca.DM(v)
#on a eu un problème avec les multiplications entre array et objets casadi, 
#on a donc converti les array de facon à ce que les ojets manipulés soient compatibl    es

q = ca.MX.sym('q', p)
r= ca.MX.sym('r', m)

f = c.T @ r - 0.5*v.T @ h(q, d1, alpha) - 0.3*v.T @ h(q, d2, alpha) - 0.2*v.T @ h(q, d3, alpha)


#contraintes
c1= -q 
c2 = -r
c3= A @ q - r

c = ca.vertcat(c1, c2, c3)


nlp = {'x': ca.vertcat(q, r), 'f': f, 'g': c}

solver = ca.nlpsol('solver', 'ipopt', nlp)


sol = solver(lbg=-ca.inf, ubg=0)

z_opt_h = sol['x']
print(z_opt_h)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:       28
Number of nonzeros in Lagrangian Hessian.............:        3

Total number of variables............................:        8
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:       13
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:       13

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0 -9.7748379e-01 0.00e+00 1.13e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [27]:
# rendement de la solution avec h
cost = ca.DM(1e-3 * np.array([30,1,1.3,4,1]))
def f(q, r):
    return cost.T @ r - 0.5*v.T @ h(q, d1, alpha) - 0.3*v.T @ h(q, d2, alpha) - 0.2*v.T @ h(q, d3, alpha)

q_opt_h= z_opt_h[:p]
r_opt_h = z_opt_h[p:]
print("q_opt:", q_opt_h)
print("r_opt:", r_opt_h)
print("espérence de revenu optimale:", -f(q_opt_h, r_opt_h))

q_opt: [406.698, 77.1392, 54.2896]
r_opt: [1632.01, 109203, 779.982, 3628.46, 655.683]
espérence de revenu optimale: 323.96


**Question 8**
 

*Question 8.a*

 On cherche à maximiser l'espérance du revenu :

$$\max_{q \geq 0,\, r \geq 0} \quad \sum_{k=1}^K \pi^k \, v^T \min(q, d^k) - c^T r$$

$$\text{s.c.} \quad r \geq Aq$$


*Question 8.b*


On considère la fonction coût non-approchée (2) avec $K$ scénarios de demande $d^k$ de probabilités $\pi^k$.
On cherche à maximiser l'espérance du revenu :

$$\max_{q \geq 0,\, r \geq 0} \quad \sum_{k=1}^K \pi^k \, v^T \min(q, d^k) - c^T r$$

$$\text{s.c.} \quad r \geq Aq$$


On réécrit le problème sous forme de minimisation :

$$\min_{q \geq 0,\, r} \quad c^T r - \sum_{k=1}^K \pi^k \, v^T \min(q, d^k) \quad \text{s.c.} \quad Aq - r \leq 0$$

Le Lagrangien associé est :

$$\mathcal{L}(q, r, \lambda) = c^T r - \sum_{k=1}^K \pi^k \, v^T \min(q, d^k) + \lambda^T(Aq - r)$$

La condition KKT sur $r$ (le Lagrangien est lisse en $r$) donne :

$$\frac{\partial \mathcal{L}}{\partial r} = c - \lambda = 0 \implies \lambda = c$$

Or $c > 0$ par hypothèse (coûts strictement positifs). La condition de complémentarité KKT impose :

$$\lambda_i \cdot \left[(Aq)_i - r_i\right] = 0 \quad \forall i$$

Comme $\lambda_i = c_i > 0$, on conclut nécessairement :

$$r_i = (Aq)_i \quad \forall i \quad \Rightarrow \quad r = Aq$$

La contrainte (1) est donc active à l'optimum.

*Question 8.c*

On remplace donc r par Aq et on cherche à maximiser la quantité :

$$\boxed{\sum_{k=1}^{K} \pi^k \, v^T h(q, d^k) - c^T Aq}$$




**Question 9**

*Question 9.a*

eccrire lagrangien, dériver // uk, dire dL/duk=0 et comme v non nul alors au moins un des deux lambdas non nuls, donc contrainte active d'ou on a bien le min 

*Question 9.b*

Le problème se reformule : 

$$\max_{0 \geq u^k - q,\, 0 \geq u^k - d^k} \; \sum_{k=1}^{K} \pi^k \, v^Tu^k \;-\; c^T Aq$$

L'intérêt principal est de s'être débarrassé du minimum dans le problème, est donc de s'être ramené à un cas lisse.

*Question 9.c*


In [2]:
alpha = 0.1
c = 1e-3 * np.array([30,1,1.3,4,1])
v = np.array([0.9,1.5,1.1])
d1 = np.array([400,67,33])
d2 = np.array([500,80,53])
d3 = np.array([300,60,43])
A = np.array([
    [3.5,2,1],
    [250,80,25],
    [0,8,3],
    [0,40,10],
    [0,8.5,0]
])

m = 5
p = 3

c = ca.DM(c)
v = ca.DM(v)
#on a eu un problème avec les multiplications entre array et objets casadi, 
#on a donc converti les array de facon à ce que les ojets manipulés soient compatibles

q = ca.MX.sym('q', p)
u1 = ca.MX.sym('u1', p)
u2 = ca.MX.sym('u2', p)
u3 = ca.MX.sym('u3', p)

f = c.T @ A@ q - 0.5*v.T @ u1-0.3*v.T @ u2 - 0.2*v.T @ u3


#contraintes
c1= -q + u1
c2 = -q + u2
c3 = -q + u3
c4 = -d1 + u1
c5 = -d2 + u2
c6 = -d3 + u3

c = ca.vertcat(c1, c2, c3, c4, c5, c6)


nlp = {'x': ca.vertcat(q, u1, u2, u3), 'f': f, 'g': c}

solver = ca.nlpsol('solver', 'ipopt', nlp)


sol = solver(lbg=-ca.inf, ubg=0)

z_opt = sol['x']
print(z_opt)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:       27
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:       12
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:       18
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:       18

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 0.00e+00 8.75e-01  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

**Question 10**

*Question 10.a*

On peut penser à une méthode de gradient proximale. 

Note : nous nous sommes servis de l'IA pour la conversion de nos notes manuscrites en markdown, car ni l'un ni l'autre ne connaissions la syntaxe exacte ; ainsi que pour les technicités de la bibliothèque casadi.